# Structural Stagnation in PRGA
Numerical illustration of the finite-accumulation phenomenon.

This notebook reproduces the experiments corresponding to Theorem 1.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Utilities

In [ ]:
def approx_P_alpha(alpha: float, K: int = 20000) -> float:
    """
    Stable numerical approximation of
    P_alpha = prod_{k=2}^∞ (1 - 1/k^alpha)
    using logarithmic summation.
    """
    k = np.arange(2, K + 1, dtype=np.float64)
    log_terms = np.log(1.0 - 1.0 / (k ** alpha))
    return float(np.exp(np.sum(log_terms)))

def make_pair_with_coherence(n: int, mu: float, rng: np.random.Generator):
    """
    Generates two unit vectors x1, x2 in R^n
    with prescribed coherence mu = <x1, x2>.
    """
    if not (0.0 <= mu < 1.0):
        raise ValueError("mu must be in [0, 1).")

    x1 = rng.standard_normal(n)
    x1 /= np.linalg.norm(x1)

    u = rng.standard_normal(n)
    u -= (u @ x1) * x1
    u /= np.linalg.norm(u)

    x2 = mu * x1 + np.sqrt(1.0 - mu**2) * u
    return x1, x2

## PRGA Implementation

In [ ]:
def prga_two_atoms(x1, x2, y, alpha: float, M: int):
    """
    Power-Relaxed Greedy Algorithm with two atoms.
    Step size: lambda_m = m^{-alpha}
    """
    f = np.zeros_like(y)
    r = y - f
    errs = [np.linalg.norm(r)]

    for m in range(1, M + 1):
        lam = m ** (-alpha)

        c1 = r @ x1
        c2 = r @ x2

        if abs(c1) >= abs(c2):
            g = np.sign(c1) * x1
        else:
            g = np.sign(c2) * x2

        f = (1.0 - lam) * f + lam * g
        r = y - f
        errs.append(np.linalg.norm(r))

    return np.array(errs)

## Sweep over coherence μ

In [ ]:
def sweep_mu(
    n=200,
    b=0.25,
    alphas=(1.1, 1.5),
    M=800,
    mu_grid=None,
    seed=0
):
    rng = np.random.default_rng(seed)
    if mu_grid is None:
        mu_grid = np.linspace(0.0, 0.95, 20)

    P = {a: approx_P_alpha(a) for a in alphas}

    results = {}
    for a in alphas:
        results[a] = {
            "mu": [],
            "min_err": [],
            "theory_lb": []
        }

    for mu in mu_grid:
        x1, x2 = make_pair_with_coherence(n, mu, rng)
        y = (1.0 - b) * x1 + b * x2

        for a in alphas:
            errs = prga_two_atoms(x1, x2, y, alpha=a, M=M)

            min_err = np.min(errs[1:])
            lb = b * (1.0 - mu) * np.sqrt((1.0 + mu) / 2.0) * P[a]

            results[a]["mu"].append(mu)
            results[a]["min_err"].append(min_err)
            results[a]["theory_lb"].append(lb)

    return results, P

## Plotting

In [ ]:
def plot_sweep(results, title=None):
    plt.figure(figsize=(8,6))

    for a, d in results.items():
        mu = np.array(d["mu"])
        val = np.array(d["min_err"])
        lb = np.array(d["theory_lb"])

        plt.plot(mu, val, marker="o", label=f"alpha={a}")
        plt.plot(mu, lb, linestyle="--", linewidth=2,
                 label=f"Theory LB (alpha={a})")

    plt.yscale("log")
    plt.xlabel("coherence μ")
    plt.ylabel("min_m ||r_m||_2")
    if title:
        plt.title(title)
    plt.legend()
    plt.grid(True, which="both", linestyle=":")
    plt.tight_layout()
    plt.show()

## Run experiment

In [ ]:
n = 200
b = 0.25
alphas = (1.1, 1.5)
M = 800
mu_grid = np.linspace(0.0, 0.95, 20)

results, P = sweep_mu(
    n=n,
    b=b,
    alphas=alphas,
    M=M,
    mu_grid=mu_grid,
    seed=0
)

print("Approximate P_alpha values:")
for a in P:
    print(f"alpha={a}: P_alpha ≈ {P[a]}")

plot_sweep(
    results,
    title="Stagnation vs coherence (PRGA, alpha=1.1 and 1.5)"
)